# Member 3: Product, Revenue & Marketing Analytics

**E-Commerce Customer & Revenue Intelligence Platform**  
Brazilian E-Commerce Public Dataset by Olist

### Analytics Pipeline Architecture:
```text
Orders + Products + Payments + Returns + Marketing
                         ↓
                Revenue Analytics
                         ↓
                Product Analytics
                         ↓
                Marketing Analytics
```

---
### Scope of Work:
1. **Revenue Analytics**:
   - Monthly Revenue & Growth (MoM %, YoY %, 3-Month Rolling Average)
   - Average Order Value (AOV) & Basket Size across Marketplace, Months, Categories & Regions
   - Revenue by Product Category & Category Share %
   - Revenue by Brazilian States & Macro-Regions
2. **Product Analytics**:
   - Best-Selling Products by Volume & Revenue (Pareto 80/20 Analysis)
   - Most Profitable Products (Gross Profit & Net Contribution Margin)
   - Low-Margin Products & High Freight Drag Products
   - Return-Heavy Products & Return Reasons Breakdown
   - Category Performance Scorecard & BCG Portfolio Matrix
3. **Marketing Analytics**:
   - Multi-Channel Campaign Performance (Impressions, Clicks, CTR, Leads, CVR)
   - Marketing Spend Allocation by Channel
   - Customer Acquisition Cost (CAC) Trends & CAC-to-AOV Ratios
   - Return on Ad Spend (ROAS) & Marketing ROI %

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

# Visual configurations
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Database connection
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', '')
DB_HOST = os.getenv('POSTGRES_HOST', 'localhost')
DB_PORT = os.getenv('POSTGRES_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'ecommerce')

engine = create_engine(f'postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
print('Connected successfully to PostgreSQL database:', DB_NAME)

## 1. Revenue Analytics
### 1.1 Monthly Revenue & Revenue Growth (MoM % and YoY %)

In [ ]:
with open('../sql/analytics/monthly_revenue_growth.sql') as f:
    query = f.read()

df_monthly = pd.read_sql(text(query), engine)
display(df_monthly[['year', 'month', 'period_label', 'total_orders', 'total_revenue', 'monthly_aov', 'mom_growth_pct', 'cumulative_revenue']].tail(12))

# Plot Monthly Revenue and MoM Growth
fig, ax1 = plt.subplots(figsize=(14, 6))

x_labels = df_monthly['period_label']
x = np.arange(len(x_labels))

color = '#1f77b4'
ax1.set_xlabel('Month', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Revenue (BRL)', color=color, fontsize=12, fontweight='bold')
bars = ax1.bar(x, df_monthly['total_revenue'], color=color, alpha=0.8, label='Monthly Revenue')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(x)
ax1.set_xticklabels(x_labels, rotation=45, ha='right')

# Rolling 3M line
ax1.plot(x, df_monthly['rolling_3m_avg_revenue'], color='#ff7f0e', linewidth=2.5, linestyle='--', label='3-Month Rolling Avg')

ax1.set_title('Monthly Revenue & 3-Month Moving Average (Sep 2016 - Sep 2018)', fontsize=14, fontweight='bold', pad=15)
ax1.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 1.2 Average Order Value (AOV) Analysis across Categories & Regions

In [ ]:
q_region_aov = """
SELECT macro_region, SUM(total_orders) AS total_orders, ROUND(SUM(total_revenue), 2) AS total_revenue,
       ROUND(SUM(total_revenue) / NULLIF(SUM(total_orders), 0), 2) AS regional_aov,
       ROUND(SUM(freight_revenue) / NULLIF(SUM(total_orders), 0), 2) AS freight_per_order,
       ROUND(100.0 * SUM(freight_revenue) / NULLIF(SUM(total_revenue), 0), 2) AS freight_share_pct
FROM ecommerce.vw_regional_revenue
GROUP BY macro_region
ORDER BY regional_aov DESC;
"""
df_reg_aov = pd.read_sql(text(q_region_aov), engine)
display(df_reg_aov)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_reg_aov, x='macro_region', y='regional_aov', palette='Blues_r', ax=ax)
ax.set_title('Average Order Value (AOV) by Brazilian Macro-Region', fontsize=14, fontweight='bold')
ax.set_ylabel('AOV (BRL)', fontsize=12)
ax.set_xlabel('Macro Region', fontsize=12)
for p in ax.patches:
    ax.annotate(f'R$ {p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.show()

### 1.3 Revenue by Category (Top Categories & Pareto Distribution)

In [ ]:
with open('../sql/analytics/revenue_by_category.sql') as f:
    df_cat_rev = pd.read_sql(text(f.read()), engine)

display(df_cat_rev.head(10))

fig, ax = plt.subplots(figsize=(12, 6))
top10_cat = df_cat_rev.head(10).sort_values('total_revenue', ascending=True)
ax.barh(top10_cat['category'], top10_cat['total_revenue'] / 1e6, color='#2ca02c')
ax.set_title('Top 10 Product Categories by Revenue (Millions BRL)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (Million BRL)', fontsize=12)
for p in ax.patches:
    ax.annotate(f'R$ {p.get_width():.2f}M', (p.get_width(), p.get_y() + p.get_height()/2.),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

## 2. Product Analytics
### 2.1 Best-Selling vs Most Profitable Products

In [ ]:
with open('../sql/analytics/product_profitability.sql') as f:
    df_prof = pd.read_sql(text(f.read()), engine)

print('Top 10 Most Profitable Products:')
display(df_prof[['rank', 'product_id', 'category', 'total_units_sold', 'total_revenue', 'gross_profit', 'net_profit_contribution', 'net_profit_margin_pct']].head(10))

fig, ax = plt.subplots(figsize=(12, 5))
top_prof = df_prof.head(8)
x = np.arange(len(top_prof))
w = 0.35
ax.bar(x - w/2, top_prof['total_revenue'], width=w, label='Total Revenue', color='#1f77b4')
ax.bar(x + w/2, top_prof['net_profit_contribution'], width=w, label='Net Profit Contribution', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels([f'{c[:12]}..\n({pid[:6]})' for c, pid in zip(top_prof['category'], top_prof['product_id'])], rotation=25)
ax.set_title('Top 8 Most Profitable Products: Revenue vs Net Profit', fontsize=14, fontweight='bold')
ax.set_ylabel('BRL', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Low-Margin & Return-Heavy Products

In [ ]:
q_ret_reasons = """
SELECT return_reason, COUNT(*) as returned_items, ROUND(SUM(total_refund), 2) as total_refund,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct_share
FROM ecommerce.vw_returns_analysis
GROUP BY return_reason
ORDER BY returned_items DESC;
"""
df_reasons = pd.read_sql(text(q_ret_reasons), engine)
display(df_reasons)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(df_reasons['returned_items'], labels=df_reasons['return_reason'], autopct='%1.1f%%', startangle=140,
       colors=['#ff9999','#66b3ff','#99ff99','#ffcc99','#c2c2f0','#ffb3e6'])
ax.set_title('Return & Cancellation Reasons Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Marketing Analytics
### 3.1 Marketing Campaign Performance, Conversion & Funnel Metrics

In [ ]:
with open('../sql/analytics/marketing_campaign_performance.sql') as f:
    df_mkt_camp = pd.read_sql(text(f.read()), engine)

display(df_mkt_camp)

# Plot CTR and CVR by Campaign
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_mkt_camp, y='campaign_name', x='overall_ctr_pct', palette='viridis', ax=ax1)
ax1.set_title('Click-Through Rate (CTR %) by Campaign', fontsize=13, fontweight='bold')
ax1.set_xlabel('CTR %')

sns.barplot(data=df_mkt_camp, y='campaign_name', x='overall_cvr_pct', palette='mako', ax=ax2)
ax2.set_title('Conversion Rate (CVR %) by Campaign', fontsize=13, fontweight='bold')
ax2.set_xlabel('CVR %')

plt.tight_layout()
plt.show()

### 3.2 Marketing Spend, CAC, ROAS & ROI % by Acquisition Channel

In [ ]:
q_mkt_channel = """
WITH channel_summary AS (
    SELECT dc.channel, SUM(fmp.marketing_spend) AS spend, SUM(fmp.orders) AS orders,
           SUM(fmp.attributed_revenue) AS revenue
    FROM ecommerce.dim_campaign dc
    JOIN ecommerce.fact_marketing_performance fmp ON dc.campaign_key = fmp.campaign_key
    GROUP BY dc.channel
)
SELECT channel, ROUND(spend, 2) AS marketing_spend, orders, ROUND(revenue, 2) AS attributed_revenue,
       ROUND(spend / orders, 2) AS cac, ROUND(revenue / spend, 2) AS roas,
       ROUND(100.0 * ((revenue * 0.30) - spend) / spend, 2) AS roi_pct
FROM channel_summary
ORDER BY revenue DESC;
"""
df_chan = pd.read_sql(text(q_mkt_channel), engine)
display(df_chan)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_chan, x='channel', y='cac', palette='rocket', ax=ax1)
ax1.set_title('Customer Acquisition Cost (CAC) by Channel (BRL)', fontsize=13, fontweight='bold')
ax1.set_ylabel('CAC (BRL)')
for p in ax1.patches:
    ax1.annotate(f'R$ {p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

sns.barplot(data=df_chan, x='channel', y='roas', palette='crest', ax=ax2)
ax2.set_title('Return on Ad Spend (ROAS Multiplier) by Channel', fontsize=13, fontweight='bold')
ax2.set_ylabel('ROAS (x)')
for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.1f}x', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Key Takeaways & Strategic Recommendations
1. **Core Revenue Drivers**: Health & Beauty, Watches & Gifts, Bed Bath Table, and Sports & Leisure drive over 32% of total marketplace revenue.
2. **Geographic Concentration**: Southeast generates 64.6% of revenue with healthy 13.1% freight burden. In contrast, North and Northeast suffer from high freight shares (17-18.5%), requiring regional fulfillment hub optimization.
3. **Return Rate & Quality**: Marketplace return rate is 2.29% of items (444.5k BRL in refunds). The primary causes are return/exchange requests (35.4%) and pre-delivery cancellations (21.0%).
4. **Marketing Efficiency**: Blended CAC sits at R$ 21.46 against an AOV of R$ 168.50 (CAC is ~12.7% of AOV). Email marketing and Brand Paid Search deliver the highest ROI (575% and 326% respectively), while Social Prospecting has the highest CAC (R$ 32.40).